# Giving the Agent a Memory

The Lab 5 agent is good at one question at a time. Ask it about `N10011`, then ask
"any maintenance events on that aircraft?" and it has no idea what *that* means.
Close the notebook and everything it worked out is gone.

This lab fixes both, and it fixes them in the graph you already have. That is the
part worth staying awake for. Most memory products store conversations in their own
database, so "which aircraft do people keep asking about" and "which aircraft is
actually breaking" live in two systems and get joined by hand, badly. Here a
remembered aircraft **is** the aircraft node from Lab 2. One `MATCH` walks from a
technician's question to a maintenance record.

**Prerequisites**

| Lab | What this notebook needs from it |
|---|---|
| [Lab 2](../Lab_2_Databricks_ETL_Neo4j) | The fleet graph in your own Aura instance |
| [Lab 1 notebook 02](../Lab_1_Aura_Setup/02_credentials_and_cypher.ipynb) | The secret scope holding your Aura credentials |
| [Lab 5](../Lab_5_LangGraph_Agent) | `tools.py`, and the agent you are about to give a memory |

Lab 5's folder must sit next to this one in your workspace. This notebook imports
its tools rather than carrying a second copy of them.

**Roughly 75 minutes**, spent about like this:

| | Minutes |
|---|---|
| Sections 1 to 3, install and connect | 15 |
| Section 4, adopting the fleet graph | 10 |
| Section 5, seeding a week of shift history | 10 |
| Section 6, the query that needs both halves | 15 |
| Sections 7 and 8, wiring recall and remember into the agent | 15 |
| Section 9, measuring what memory bought | 10 |

## Section 1: Install

`neo4j-agent-memory` is not on the cluster by default. It is installed from a
wheel on a Unity Catalog volume rather than from PyPI, and the reason is written
into `memory.py` if you want it: the released version drops most of the
`MENTIONS` edges that Section 6 depends on, silently, with no error.

`dbutils.library.restartPython()` restarts the Python process. Every variable you
have defined disappears, which is why this is the first cell and not the third.

In [ ]:
%pip install /Volumes/databricks-neo4j-workshop/aircraft/raw_data/neo4j_agent_memory-0.5.1.dev1+mentions-py3-none-any.whl httpx>=0.27.0

In [ ]:
dbutils.library.restartPython()

## Section 2: Configuration

The same two values Lab 5 asked for. Your secret scope is derived from
`current_user()`, so there is nothing to copy across from the other notebook.

In [ ]:
# ==================================================
# CONFIGURATION - replace GENIE_AGENT_ID with yours
# ==================================================

# From Lab 4 Part A, the same value you used in Lab 5. Open your Genie space and
# take the ID out of the URL:
#   https://<workspace>/genie/rooms/<GENIE_AGENT_ID>
GENIE_AGENT_ID = ""

# From your Genie space settings, the same value you typed in Lab 5 notebook 02.
# Section 10 declares it as a resource, and without it the redeployed endpoint
# routes sensor questions correctly and then cannot run the SQL.
WAREHOUSE_ID = ""

# Who you are, for the memory. The headline query in Section 6 counts distinct
# technicians, so this is what makes you the sixth one.
MY_USER_ID = "tech:you"

import sys

sys.path.insert(0, ".")

from memory import ensure_labs_on_path

lab3_dir, lab5_dir = ensure_labs_on_path()

from data_utils import read_neo4j_secrets, secret_scope_name

SECRET_SCOPE = secret_scope_name(spark)

# Same scope and same key as Lab 3 and Lab 5. A scope written before the
# database key existed reads back as the AuraDB Free default.
NEO4J_DATABASE = read_neo4j_secrets(dbutils, SECRET_SCOPE)["database"]

print(f"Secret scope: {SECRET_SCOPE}")
print(f"Lab 3 code:   {lab3_dir}")
print(f"Lab 5 code:   {lab5_dir}")
if not GENIE_AGENT_ID:
    print("\nGENIE_AGENT_ID is empty. Sections 7 onward need it.")
if not WAREHOUSE_ID:
    print("WAREHOUSE_ID is empty. Section 10 needs it.")

## Section 3: Connecting the memory client

Two things happen in the next cell, and only one of them is obvious.

**The obvious one.** A `MemoryClient` connects to the same Aura instance holding
your fleet graph. Not a second database. The same one.

**The less obvious one.** `neo4j-agent-memory` needs an embedding model and a
chat model, and it has never heard of Databricks. It does not need to. It asks
for three `Protocol`s, and `memory.py` supplies two small classes that satisfy
them by calling the Foundation Model endpoints you have used since Lab 3.

You may notice Lab 3 already has classes called `DatabricksEmbeddings` and
`DatabricksLLM`. These are not those. Lab 3's pair satisfies `neo4j-graphrag`,
whose interface is synchronous; this pair satisfies `neo4j-agent-memory`, whose
Protocols are `async`. Two libraries, two shapes, the same two endpoints
underneath. They are named `MemoryEmbeddings` and `MemoryLLM` so you never have
to work out which is which.

**First connect is slow, about 20 seconds.** The library is creating 25 indexes
and 9 constraints, five of them vector indexes sized from the embedding model's
1024 dimensions. Its full schema is 33 and 12; `memory.py` names the four
subsystems this lab never writes to and the library leaves those out. It happens
once per database, not once per session.

In [ ]:
from memory import EMBEDDING_ENDPOINT, LLM_ENDPOINT, MemorySession

# The password is read, used, and dropped inside this call, so it never lands in
# a notebook variable. Same scope and same keys as Lab 3 and Lab 5.
session = MemorySession.open_from_secrets(
    dbutils, SECRET_SCOPE, database=NEO4J_DATABASE
)

# The database name came out of a secret, so printing it prints [REDACTED].
print("Memory client connected")
print(f"  embeddings: {EMBEDDING_ENDPOINT}")
print(f"  llm:        {LLM_ENDPOINT}")

### An aside on `async`

`neo4j-agent-memory` is async from top to bottom, and a notebook cell is not.
The usual bridge, `asyncio.run(...)` per call, breaks here: it opens and closes
an event loop each time, and the Neo4j driver inside the client binds to the
loop that created it. The next cell would get a driver attached to a loop that
no longer exists.

So `MemorySession` keeps one event loop alive on a background thread for the
whole notebook. `session.run(coro)` hands work to it and blocks until the answer
comes back, which is why everything below reads like ordinary code.

## Section 4: Adopting the fleet graph

This is the step that makes the lab work.

Right now your graph has `(:Aircraft {tail_number: 'N10004'})`. The memory
library wants to attach conversations to `(:Entity {name, type})` nodes. Left
alone it would create its own `N10004` beside yours, and you would be back to
two systems and a manual join.

**Adoption** stops that. It adds the `:Entity` label and the library's `id`,
`type` and `name` properties to nodes you already have. Afterwards there is one
`N10004`: still an `Aircraft` with its flights and maintenance events, and now
also an `Entity` that messages can point at.

### Dry run first. Always.

Adoption writes. Run the dry run, read the counts, and only then commit.

In [ ]:
from memory import ADOPT_LABEL_TO_TYPE, ADOPT_NAME_PROPERTY, adoption_dry_run, describe_adoption_report

print(f"Would adopt: {ADOPT_LABEL_TO_TYPE}")
print(f"Name from:   {ADOPT_NAME_PROPERTY}\n")

report = adoption_dry_run(session)
print(describe_adoption_report(report))
print(f"\ndry_run={report.dry_run}, nothing was written")

### Why only `Aircraft`?

The obvious move is to adopt everything: `System`, `Component`, `Sensor`, the
lot. Richer memory, surely.

It destroys your graph, and nothing tells you.

Adoption sets `n.type` unconditionally. Your `Component` nodes already have a
`type`, holding values like `Turbine`. Adopt them and every one of those becomes
the string `COMPONENT`. No error, no warning. The Lab 2 and Lab 4 queries that
filter on `type` quietly return nothing, and the only way back is a full reload.

This was found the expensive way while building this lab. `memory.py` now
refuses those four labels outright. Run the next cell to see it refuse.

In [ ]:
from memory import DESTRUCTIVE_ADOPTION_LABELS

print("Labels adoption would damage:\n")
for label, (count, holds) in sorted(DESTRUCTIVE_ADOPTION_LABELS.items()):
    print(f"  {label:<10} {count:>4} nodes, type currently holds {holds}")

try:
    adoption_dry_run(session, label_to_type={"Aircraft": "AIRCRAFT", "Component": "COMPONENT"})
except ValueError as error:
    print(f"\nRefused, as it should be:\n\n{error}")

### Commit the adoption

36 aircraft, about three seconds. It is idempotent: run it twice and the second
run reports them as already adopted rather than doing anything.

In [ ]:
from memory import adopt_aircraft

report = adopt_aircraft(session)
print(describe_adoption_report(report))
print(f"\nmigrated={report.total_migrated}  already_adopted={report.total_already_adopted}")

In [ ]:
# Proof: one node, both labels, fleet properties and memory properties together.
rows = session.cypher(
    """
    MATCH (ac:Aircraft:Entity {tail_number: $tail})
    RETURN labels(ac) AS labels, ac.tail_number AS tail_number,
           ac.model AS model, ac.name AS memory_name, ac.type AS memory_type
    """,
    {"tail": "N10004"},
)
for row in rows:
    print(row)

## Section 5: A week of shift history

Memory is only interesting once there is some. Rather than have you type ten
messages, the next cell replays a week of maintenance shift conversation: five
technicians, five sessions, ten messages between them.

### Explicit mentions, and why they are the point

When you write a message, the library can work out which entities it mentions in
two ways.

**Automatic.** An LLM reads the message and extracts entities. General, and it
costs an extra model call per message, about 9.2 seconds each here.

**Explicit.** You hand it the entities. About 5.6 seconds a message, and exact.

Explicit wins for an agent, and not mainly on speed. An agent normally *already
knows* which entities it touched. It just queried `N10011`. Paying a language
model to rediscover a fact you already have is the kind of thing that looks
reasonable in a demo and looks silly in production.

Here the extraction is a regular expression, because tail numbers have a format:

```python
TAIL_NUMBER_RE = re.compile(r"\bN\d{5}\b")
```

In [ ]:
from memory import SEED_MESSAGES, aircraft_mentions

sample = SEED_MESSAGES[1].content
print(f"Message:  {sample}\n")
print("Mentions:", [(ref.name, ref.type, ref.label) for ref in aircraft_mentions(sample)])

### Write it

Ten messages, one at a time, about a minute in total.

One at a time is deliberate. There is a batch API, `add_messages_batch`, and it
is faster. It also takes no `explicit_mentions` argument at all, so batching
would mean either paying for automatic extraction or getting no mentions. The
mentions are the entire lab. We take the minute.

In [ ]:
from memory import seed_memory

def show(index, total, mentions):
    print(f"  [{index:>2}/{total}] {mentions} mention(s) linked")

linked = seed_memory(session, on_progress=show)
print(f"\n{linked} mentions linked across {len(SEED_MESSAGES)} messages")

In [ ]:
# What that built. Note the last row: Message nodes point at your Aircraft nodes.
# Written as a top-level UNION rather than a CALL subquery: session.cypher()
# rejects `CALL {`, because a subquery is allowed to write and the guard cannot
# tell that this one does not.
rows = session.cypher(
    """
    MATCH (n:User)         RETURN 'User' AS label, count(n) AS count
    UNION
    MATCH (n:Conversation) RETURN 'Conversation' AS label, count(n) AS count
    UNION
    MATCH (n:Message)      RETURN 'Message' AS label, count(n) AS count
    """
)
for row in rows:
    print(f"  {row['label']:<14} {row['count']}")

edges = session.cypher(
    "MATCH (:Message)-[r:MENTIONS]->(:Aircraft) RETURN count(r) AS mentions_to_aircraft"
)
print(f"  {'MENTIONS':<14} {edges[0]['mentions_to_aircraft']} -> :Aircraft")

## Section 6: The query that needs both halves

Here is the payoff. Three queries over the same graph.

**Query one: the fleet graph alone.** This is what a maintenance dashboard shows
you. Rank aircraft by how many critical events they have logged.

In [ ]:
from memory import FLEET_ONLY_QUERY

for row in session.cypher(FLEET_ONLY_QUERY, {"limit": 6}):
    print(f"  {row['aircraft']}  events={row['events']:<3} critical={row['critical']}")

**Query two: the conversation memory alone.** This is what a memory product with
no domain graph shows you. Rank aircraft by how many different technicians have
asked about them.

In [ ]:
from memory import MEMORY_ONLY_QUERY

for row in session.cypher(MEMORY_ONLY_QUERY, {"limit": 6}):
    print(f"  {row['aircraft']}  technicians={row['technicians']}  mentions={row['mentions']}")

### Find `N10011` in both lists

It is near the bottom of the first and at the top of the second.

Read separately, each list is unremarkable. An aircraft with few critical events
is fine. An aircraft several people asked about is a busy week. Put them side by
side and it is a different sentence: **three technicians independently pulled the
EGT trend on an aircraft the maintenance record says is healthy.** Either they
are seeing something the record has not caught yet, or three people just wasted a
shift each on the same dead end. Both are worth a supervisor's attention, and
neither list says it alone.

Now the query that says it. Watch line 8.

In [ ]:
from memory import HEADLINE_QUERY

print(HEADLINE_QUERY)

In [ ]:
for row in session.cypher(HEADLINE_QUERY, {"min_technicians": 2}):
    print(
        f"  {row['aircraft']}  asked_by={row['technicians']} {row['asked_by']}\n"
        f"           events={row['events']} critical={row['critical']} systems={row['systems']}\n"
    )

### What made that one query instead of two

```cypher
MATCH (u:User)-[:HAS_CONVERSATION]->(c)-[:HAS_MESSAGE]->(m)-[:MENTIONS]->(ac:Aircraft)
WITH ac, count(DISTINCT u) AS technicians, ...
MATCH (ac)<-[:AFFECTS_AIRCRAFT]-(ev:MaintenanceEvent)-[:AFFECTS_SYSTEM]->(sys:System)
```

`ac` is bound in the memory half and reused in the fleet half. **Same node.**

Without adoption, `ac` on line 1 would be a memory `Entity` that happens to share
a name with your `Aircraft`, and joining them would mean exporting both sides and
matching strings in Python. That code exists in a lot of production systems. It
is where the tail number `N10011` and the tail number `n10011 ` go to disagree.

Take five minutes and try one of these:

- Which technicians asked about aircraft that later had a `CRITICAL` event?
- Which *systems* draw questions that the maintenance record disagrees with?
- Is any aircraft mentioned by nobody and failing constantly?

In [ ]:
# Your turn.
session.cypher(
    """
    MATCH (u:User)-[:HAS_CONVERSATION]->(:Conversation)-[:HAS_MESSAGE]->(m:Message)
          -[:MENTIONS]->(ac:Aircraft)
    RETURN ac.tail_number AS aircraft, collect(DISTINCT u.identifier) AS who
    ORDER BY aircraft
    """
)

## Section 7: Recall and remember

Time to give the Lab 5 agent this memory. Two new nodes, either side of the
supervisor it already has:

```
  question
     |
     v
  +----------+     what has this team said before?
  |  recall  |     semantic search over past messages
  +----------+
     |
     v
  +------------+
  | supervisor |<--------+     Lab 5's, one extra prompt variable and one
  +------------+         |     extra field in the JSON it replies with
     |                   |
     +---> genie / cypher / graphrag
     |
     v
  +-----------+
  | synthesize|
  +-----------+
     |
     v
  +----------+     write the question and the answer back,
  | remember |     with the aircraft they mention
  +----------+
     |
     v
   answer
```

`recall` runs once per question, not once per tool call. It costs 3 to 5 seconds.

In [ ]:
# Warm up: what does recall actually find?
from memory import build_recall_node

recall_node = build_recall_node(session)

print(recall_node({"question": "what is the EGT margin doing on N10011?"})["recalled"])

Note that the question above never mentions a technician or a shift, and recall
still finds the three separate conversations about that aircraft's EGT margin.
That is semantic search over message content, not a keyword match.

You may see a Neo4j deprecation warning about `db.index.vector.queryNodes`. It
works on Aura's current version; the library has not moved off it yet.

In [ ]:
# Build the Lab 5 agent, plus the two memory nodes.
from databricks.sdk import WorkspaceClient
from langgraph.graph import END, START, StateGraph

from tools import (
    LLM_ENDPOINT,
    TOOL_NAMES,
    build_cypher_node,
    build_genie_node,
    build_graphrag_node,
    build_synthesize_node,
    get_embedder,
    get_llm,
    open_driver_from_secrets,
    route_from_supervisor,
)
from memory import (
    EMBEDDING_ENDPOINT,
    MemoryAgentState,
    build_memory_supervisor_node,
    build_remember_node,
)

driver = open_driver_from_secrets(dbutils, SECRET_SCOPE)
llm = get_llm(LLM_ENDPOINT)
embedder = get_embedder(EMBEDDING_ENDPOINT)

genie_node = build_genie_node(GENIE_AGENT_ID, WorkspaceClient())
cypher_node = build_cypher_node(driver, llm, database=NEO4J_DATABASE)
graphrag_node = build_graphrag_node(driver, llm, embedder, database=NEO4J_DATABASE)

available = tuple(
    name for name in TOOL_NAMES
    if name != "graphrag_node" or getattr(graphrag_node, "available", False)
)

builder = StateGraph(MemoryAgentState)
builder.add_node("recall", recall_node)
builder.add_node("supervisor", build_memory_supervisor_node(llm, available_tools=available))
builder.add_node("genie_node", genie_node)
builder.add_node("cypher_node", cypher_node)
builder.add_node("graphrag_node", graphrag_node)
builder.add_node("synthesize", build_synthesize_node(llm))
builder.add_node("remember", build_remember_node(session, default_user=MY_USER_ID))

builder.add_edge(START, "recall")
builder.add_edge("recall", "supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "genie_node": "genie_node",
        "cypher_node": "cypher_node",
        "graphrag_node": "graphrag_node",
        "synthesize": "synthesize",
    },
)
for tool_name in TOOL_NAMES:
    builder.add_edge(tool_name, "supervisor")
builder.add_edge("synthesize", "remember")
builder.add_edge("remember", END)

agent = builder.compile()
print(f"Compiled. Tools available: {available}")

In [ ]:
def ask(question, session_id="lab6-session-01", user=None):
    """Ask the memory-enabled agent one question."""
    state = agent.invoke({
        "question": question,
        "trace": [],
        "findings": [],
        "session_id": session_id,
        "user_identifier": user or MY_USER_ID,
    })
    print(f"Q: {question}")
    # state["question"] is not always the question that went in. The supervisor
    # rewrites it once, from memory, and this is where you see that happen.
    if state.get("asked"):
        print(f"   resolved:  {state['question']}")
    print(f"   route:     {' -> '.join(state.get('trace', [])) or '(straight to synthesis)'}")
    print(f"   remembered: {state.get('remembered', 0)} mention(s)\n")
    print(state.get("answer", ""))
    return state

## Section 8: The thing Lab 5 could not do

Two questions. The second one contains the word *that* and nothing else to go on.

In [ ]:
first = ask("What does the maintenance history look like for N10011?")

In [ ]:
# No tail number anywhere in this question.
second = ask("Are there any vibration readings I should worry about on that aircraft?")

The Lab 5 agent answers the second question by asking a tool about "that
aircraft" and getting nothing useful back. Watch the `resolved:` line above:
the question that reached the tools names N10011, and the participant never
typed it.

Two nodes had to cooperate for that. `recall` found the first exchange before
the supervisor ever saw the question, and the supervisor rewrote the question
from it. Recall alone is not enough, because the tools are handed the question
text: memory that only changes the route still sends the word "that" to Genie,
and Genie asks which aircraft you mean.

### Continuity across sessions

The `session_id` is what makes this survive. Ask something in a fresh session and
memory still carries across, because `recall` searches every session, not just
the current one. That is the difference between a chat window with scrollback and
an agent that knows what the team has been working on.

In [ ]:
third = ask(
    "Which aircraft has the team been most concerned about lately?",
    session_id="lab6-session-02",
)

In [ ]:
# You are now in the memory too. Check.
for row in session.cypher(
    """
    MATCH (u:User {identifier: $me})-[:HAS_CONVERSATION]->(c:Conversation)
          -[:HAS_MESSAGE]->(m:Message)
    RETURN c.session_id AS session, count(m) AS messages
    ORDER BY session
    """,
    {"me": MY_USER_ID},
):
    print(f"  {row['session']:<20} {row['messages']} messages")

## Section 9: What did memory actually buy?

"It seems better" is not a measurement. Build the same agent twice, once without
the memory nodes, and run both over questions that need context.

The comparison is only fair because everything else is identical: same tools,
same model, same routing rule, same synthesis prompt. What differs is the two
nodes either side of the supervisor, and the supervisor's own prompt, which
carries the recalled messages and asks for the resolved question. Nothing else
changes.

In [ ]:
from tools import build_supervisor_node

# The Lab 5 agent, rebuilt here so the comparison is like for like.
plain = StateGraph(MemoryAgentState)
plain.add_node("supervisor", build_supervisor_node(llm, available_tools=available))
plain.add_node("genie_node", genie_node)
plain.add_node("cypher_node", cypher_node)
plain.add_node("graphrag_node", graphrag_node)
plain.add_node("synthesize", build_synthesize_node(llm))
plain.add_edge(START, "supervisor")
plain.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "genie_node": "genie_node",
        "cypher_node": "cypher_node",
        "graphrag_node": "graphrag_node",
        "synthesize": "synthesize",
    },
)
for tool_name in TOOL_NAMES:
    plain.add_edge(tool_name, "supervisor")
plain.add_edge("synthesize", END)
memory_off = plain.compile()

print("Two agents compiled: memory_off and agent")

In [ ]:
# Each pair is a setup question followed by one that depends on it.
EVAL_PAIRS = [
    (
        "What does the maintenance history look like for N10011?",
        "Are there any vibration readings I should worry about on that aircraft?",
        "N10011",
    ),
    (
        "Tell me about the EGT exceedance on N10004.",
        "Which system was that on?",
        "N10004",
    ),
]

# Every tail number these questions could land on. "N10004 appears somewhere
# in the answer" is too weak a test on its own: an answer that is confidently
# about the wrong aircraft can still mention the right one in passing, and
# that is the failure worth catching, not the one worth scoring as a pass.
TAIL_NUMBERS = {"N10000", "N10004", "N10011", "N10021"}

def resolved(answer, expected):
    """True when the answer named the right aircraft and no other one."""
    wrong = {tail for tail in TAIL_NUMBERS - {expected} if tail in answer}
    return expected in answer and not wrong

def run_pair(graph, setup, followup, session_id):
    base = {"trace": [], "findings": [], "session_id": session_id, "user_identifier": MY_USER_ID}
    graph.invoke({**base, "question": setup})
    return graph.invoke({**base, "question": followup})

rows = []
for index, (setup, followup, expected) in enumerate(EVAL_PAIRS):
    off = run_pair(memory_off, setup, followup, f"eval-off-{index}")
    on = run_pair(agent, setup, followup, f"eval-on-{index}")
    rows.append({
        "followup": followup,
        "expected": expected,
        "off_resolved": resolved(off.get("answer", ""), expected),
        "on_resolved": resolved(on.get("answer", ""), expected),
        "off_answer": off.get("answer", ""),
        "on_answer": on.get("answer", ""),
    })

for row in rows:
    print(f"Q: {row['followup']}")
    print(f"   expected the answer to name {row['expected']}")
    print(f"   memory off: {'resolved' if row['off_resolved'] else 'did NOT resolve'}")
    print(f"   memory on:  {'resolved' if row['on_resolved'] else 'did NOT resolve'}\n")

resolved_off = sum(r["off_resolved"] for r in rows)
resolved_on = sum(r["on_resolved"] for r in rows)
print(f"Referring questions resolved: {resolved_off}/{len(rows)} off, {resolved_on}/{len(rows)} on")

### Read the answers, not just the score

A count of two is not a benchmark, and this notebook is not claiming one. Print
the actual text and judge it. The failure mode to look for is not a wrong answer,
it is a *confident* answer to a question the agent did not understand.

In [ ]:
for row in rows:
    print("=" * 70)
    print(f"Q: {row['followup']}\n")
    print(f"--- memory off ---\n{row['off_answer'][:600]}\n")
    print(f"--- memory on  ---\n{row['on_answer'][:600]}\n")

### What this costs

Memory is not free, and the honest version of this lab says so:

| | Cost |
|---|---|
| `recall`, once per question | 3 to 5 seconds |
| `remember`, two messages written | about 11 seconds |
| Nodes added per exchange | 2 `Message`, plus `MENTIONS` edges |

About 15 seconds of latency per question. Worth it for a shift-handover agent
where the questions are few and the context is everything. Not worth it for a
high-volume endpoint answering one-shot lookups.

`recall` runs once per question rather than once per tool call, which is the
single decision that keeps this from being three times worse.

## Section 10: Redeploying the endpoint

Lab 5 deployed this agent to a Model Serving endpoint. You are redeploying the
same endpoint, now with memory, rather than creating a second one.

The serving container has no `dbutils` and no notebook user. Lab 5's `agent.py`
already solved that: the endpoint carries `NEO4J_URI`, `NEO4J_USERNAME` and
`NEO4J_PASSWORD` as `{{secrets/<scope>/<key>}}` references, resolved by the
serving control plane. Memory reads those same three through
`MemorySession.open_from_env()`, so this redeploy needs no new secrets.

The cell below writes the module MLflow logs. It is generated here rather than
checked in beside the notebook so that you can read the whole model in one screen.

In [ ]:
%%writefile memory_agent.py
"""The Lab 6 agent, served.

Lab 5's ``agent.py`` with a recall node before the supervisor and a remember
node after synthesis. The credential handling, the Responses API plumbing and
the deferred-build behaviour are all inherited from ``FleetOpsAgent``; only the
graph wiring is new, and it is written out rather than patched into the
inherited one so that what runs in the container is what you can read here.
"""

from __future__ import annotations

import os
from typing import Any

import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.models import set_model

from agent import (
    ENV_NEO4J_DATABASE,
    ENV_NEO4J_PASSWORD,
    ENV_NEO4J_URI,
    ENV_NEO4J_USERNAME,
    AgentRuntime,
    FleetOpsAgent,
    resolve_database,
)
from memory import (
    MemoryAgentState,
    MemorySession,
    build_memory_supervisor_node,
    build_recall_node,
    build_remember_node,
)
from tools import (
    EMBEDDING_ENDPOINT,
    LLM_ENDPOINT,
    MAX_TOOL_CALLS,
    TOOL_NAMES,
    build_cypher_node,
    build_genie_node,
    build_graphrag_node,
    build_neo4j_driver,
    build_synthesize_node,
    get_embedder,
    get_llm,
    route_from_supervisor,
)

# Module-level so the session outlives a request and the 20-second index
# creation happens once per container rather than once per question.
_SESSION: MemorySession | None = None


def build_memory_runtime(config: dict[str, Any]) -> AgentRuntime:
    """Open the connections and wire the memory-enabled graph."""
    global _SESSION
    from langgraph.graph import END, START, StateGraph

    genie_agent_id = config.get("genie_agent_id") or ""
    if not genie_agent_id:
        raise RuntimeError("No 'genie_agent_id' in the model config.")
    missing = [
        name
        for name in (ENV_NEO4J_URI, ENV_NEO4J_USERNAME, ENV_NEO4J_PASSWORD)
        if not os.environ.get(name)
    ]
    if missing:
        raise RuntimeError(f"Missing environment variables: {', '.join(missing)}.")

    driver = build_neo4j_driver(
        os.environ[ENV_NEO4J_URI],
        os.environ[ENV_NEO4J_USERNAME],
        os.environ[ENV_NEO4J_PASSWORD],
    )
    # The model config wins when it names a database, then the environment
    # variable the endpoint carries from the secret scope, then the instance.
    database = resolve_database(
        driver,
        config.get("neo4j_database") or os.environ.get(ENV_NEO4J_DATABASE, ""),
    )

    # Memory reads the same three environment variables the tools do, so the
    # endpoint needs no secrets beyond the ones Lab 5 already deployed.
    if _SESSION is None:
        _SESSION = MemorySession.open_from_env(database=database)

    llm = get_llm(config.get("llm_endpoint", LLM_ENDPOINT))
    embedder = get_embedder(config.get("embedding_endpoint", EMBEDDING_ENDPOINT))

    genie_node = build_genie_node(genie_agent_id, WorkspaceClient())
    cypher_node = build_cypher_node(driver, llm, database=database)
    graphrag_node = build_graphrag_node(
        driver, llm, embedder, database=database, top_k=config.get("top_k", 3)
    )
    available = tuple(
        name
        for name in TOOL_NAMES
        if name != "graphrag_node" or getattr(graphrag_node, "available", False)
    )

    builder = StateGraph(MemoryAgentState)
    builder.add_node("recall", build_recall_node(_SESSION))
    builder.add_node(
        "supervisor",
        build_memory_supervisor_node(
            llm,
            available_tools=available,
            max_tool_calls=config.get("max_tool_calls", MAX_TOOL_CALLS),
        ),
    )
    builder.add_node("genie_node", genie_node)
    builder.add_node("cypher_node", cypher_node)
    builder.add_node("graphrag_node", graphrag_node)
    builder.add_node("synthesize", build_synthesize_node(llm))
    builder.add_node("remember", build_remember_node(_SESSION))

    builder.add_edge(START, "recall")
    builder.add_edge("recall", "supervisor")
    builder.add_conditional_edges(
        "supervisor",
        route_from_supervisor,
        {
            "genie_node": "genie_node",
            "cypher_node": "cypher_node",
            "graphrag_node": "graphrag_node",
            "synthesize": "synthesize",
        },
    )
    for name in TOOL_NAMES:
        builder.add_edge(name, "supervisor")
    builder.add_edge("synthesize", "remember")
    builder.add_edge("remember", END)

    return AgentRuntime(
        graph=builder.compile(),
        driver=driver,
        database=database,
        available_tools=available,
    )


class MemoryFleetOpsAgent(FleetOpsAgent):
    """The Lab 5 served agent, with a memory either side of its supervisor."""

    def _ensure_runtime(self):
        if self._runtime is None and not self._build_error:
            try:
                self._runtime = build_memory_runtime(self._config.to_dict())
            except Exception as error:  # noqa: BLE001 - returned to the caller
                self._build_error = f"{type(error).__name__}: {error}"
        return self._runtime


mlflow.langchain.autolog()

AGENT = MemoryFleetOpsAgent()
set_model(AGENT)

### Log and redeploy

Three things this call must get right, and all three are easy to miss.

**`resources` is the same list Lab 5 declared.** Memory adds no Databricks
resource, because it writes to Aura, and Aura is a credential rather than a
resource. Leave the argument off entirely, though, and the redeployed endpoint
loses the Genie space and the warehouse that version 1 had.

**`code_paths` needs `memory.py`.** The container gets no workspace files it was
not handed. Miss it and the endpoint fails at first request, not at build time.

**`pip_requirements` needs the wheel path.** The version `0.5.1.dev1+mentions`
resolves from no index anywhere, so MLflow's inferred requirements cannot install
it. The volume path has to be named explicitly.

In [ ]:
import mlflow

from agent import (
    DEFAULT_CONFIG,
    UC_MODEL_NAME,
    build_resources,
    endpoint_name,
    serving_environment_vars,
)
from memory import WHEEL_PATH

ENDPOINT_NAME = endpoint_name(SECRET_SCOPE)
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="lab6-memory-agent"):
    logged = mlflow.pyfunc.log_model(
        name="memory_agent",
        python_model="memory_agent.py",
        # The database name is not logged here. It came out of a secret, and
        # the endpoint reads it from NEO4J_DATABASE instead.
        model_config={**{k: v for k, v in DEFAULT_CONFIG.items()},
                      "genie_agent_id": GENIE_AGENT_ID},
        # The same list Lab 5 notebook 02 declared. Memory adds no
        # Databricks resource: it writes to Aura, which is a credential.
        resources=build_resources(GENIE_AGENT_ID, WAREHOUSE_ID),
        code_paths=[
            "memory.py",
            f"{lab5_dir}/tools.py",
            f"{lab5_dir}/agent.py",
            f"{lab3_dir}/data_utils.py",
        ],
        pip_requirements=[
            "mlflow",
            "langgraph",
            "neo4j",
            "neo4j-graphrag",
            "databricks-sdk",
            "httpx>=0.27.0",
            WHEEL_PATH,
        ],
        # The same registered model Lab 5 created. A new version of it, not a
        # new model: the endpoint below is updated to point at this version.
        registered_model_name=UC_MODEL_NAME,
    )

MODEL_VERSION = logged.registered_model_version
print(f"Registered {UC_MODEL_NAME} version {MODEL_VERSION}")
print(f"Endpoint to update: {ENDPOINT_NAME}")

Registering and deploying is the same call Lab 5 notebook 02 made. Point it at
the endpoint that lab created and it updates in place rather than making a second
one, which matters: an endpoint per participant is already the tightest quota in
this workshop.

`endpoint_name` is what does the pointing. Leave it off and `agents.deploy`
invents a name from the model, which is how you end up with two endpoints and a
quota error.

```python
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,                   # the same name Lab 5 registered
    MODEL_VERSION,                   # the version the cell above just made
    endpoint_name=ENDPOINT_NAME,     # the endpoint Lab 5 created
    environment_vars=serving_environment_vars(SECRET_SCOPE),
    scale_to_zero=True,
    tags={"lab": "6", "memory": "on"},
)
```

Redeploying takes 10 to 15 minutes. Do not wait for it. Section 11 is worth more.

## Section 11: What you built, and what to be careful with

**What you built.** An agent whose memory lives in the same graph as its domain
data, so a question about who is worried about what and a question about what is
actually breaking are the same query.

**Three things worth carrying out of here.**

*Adoption is a one-way door.* It writes to nodes you already have. Dry run,
read the counts, then commit. And know which properties the library will
overwrite: this lab lost an afternoon to `Component.type`.

*Explicit beats automatic when you already know the answer.* Your agent knows
which entities its tools touched. Telling the memory is cheaper and exact.
Reserve LLM extraction for text nobody has parsed yet.

*Measure it, including the cost.* Memory added about 15 seconds a question here.
That is a good trade for a handover agent and a bad one for a lookup endpoint.
The number is what makes it a decision rather than a preference.

**Where this goes next.** The library holds more than this lab uses: preferences
that persist per user, temporal invalidation so a correction supersedes what it
corrects rather than contradicting it, and reasoning traces that record which
tool call touched which entity. `02_instructor_demos.ipynb` shows all three.

In [ ]:
# Close the memory client and stop its background loop.
session.close()
driver.close()
print("Closed.")